## Modular Pipeline Integration Test

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import torch
import os
import json
import sys
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image


# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)


# Pipeline imports
from src.pipeline.data_ingestion import load_csv, build_image_label_df
from src.pipeline.preprocessing import preprocess_data
from src.pipeline.feature_engineering import generate_embeddings
from src.pipeline.datasets import ProductDataset
from src.pipeline.evaluation import evaluate_model
from src.pipeline.inference import predict
from src.initialization import load_environment

# Utility imports
from src.utils.vector_db_utils import VectorDBManager
from src.utils.data_cleaning import clean_stock_code

# Script imports
from src.scripts.train_cnn_from_scratch import main

# Model imports
from src.models.cnn_model import CNNModel

load_environment()

### Data Ingestion & Cleaning
Load raw data from CSV, drop duplicates, apply cleaning/ preprocessing

In [ ]:
raw_df = load_csv('../src/data/dataset/dataset.csv')
raw_df = raw_df.drop_duplicates()

# Clean and preprocess
cleaned_df = preprocess_data(raw_df)

# For vector DB: deduplicate by stockcode
cleaned_df_vdb = cleaned_df.drop_duplicates(subset='StockCode', keep='last')

# For classification: remove rare classes
counts = cleaned_df['StockCode'].value_counts()
filtered_df = cleaned_df[cleaned_df['StockCode'].isin(counts[counts > 1].index)]

filtered_df.head()

### Feature Engineering & Embedding Generation
Generate feature embeddings for the deduplicated product data (for vector DB)

In [ ]:
api_key = os.getenv('PINECONE_API_KEY')

vector_db_manager = VectorDBManager(api_key=api_key)
vector_db_manager.initialize()

embeddings = generate_embeddings(cleaned_df_vdb, vector_db_manager)

embeddings_df = pd.DataFrame(np.array(embeddings))
embeddings_df.head()

### Model Training
Training the model with scrapped data

In [ ]:
# Load and clean the CNN data
cnn_df = pd.read_csv('../src/data/dataset/CNN_Model_Train_Data.csv')
ecommerce_df = pd.read_csv('../src/data/dataset/cleaned_ecommerce_data.csv')
cnn_df = clean_stock_code(cnn_df)

In [ ]:
# Merge product descriptions (optional, for richer metadata)
product_descriptions = (ecommerce_df[['StockCode', 'Description']]
                        .drop_duplicates()
                        .dropna(subset=['Description']))
merged_df = pd.merge(cnn_df, product_descriptions, on='StockCode', how='left')
merged_df

In [ ]:
static_dir = Path('../static/images')
static_dir.mkdir(parents=True, exist_ok=True)
for stock_code in merged_df['StockCode'].unique():
    product_dir = static_dir / stock_code
    product_dir.mkdir(exist_ok=True)

In [ ]:
# Run the training script
try:
    history = main(
        project_root_str=project_root,
        batch_size=32,
        num_epochs=50,
        learning_rate=0.001
    )
    display("\nTraining completed successfully!")
    display(f"Best validation accuracy: {history['best_val_acc']:.2f}%")
except Exception as e:
    display(f"Error occurred: {str(e)}")
    raise

### Evaluation

In [ ]:
with open('C:/Users/don/dev/ds_test/ds_task_1ab/models/stockcode_to_index.json') as f:
    stockcode_to_index = json.load(f)

In [ ]:
images_root = r'C:\Users\don\dev\ds_test\ds_task_1ab\model_test'
df = build_image_label_df(images_root, stockcode_to_index)
df

In [ ]:
# Prepare the dataset and DataLoader
image_paths = df['image_path']  # List of image paths for each row in train_df
labels = df['label']       # List of class indices for each row in train_df

train_dataset = ProductDataset(image_paths, labels, transform=None)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

# Load the trained model
num_classes = len(set(labels))
model = CNNModel(num_classes)
model.load_state_dict(torch.load('C:/Users/don/dev/ds_test/ds_task_1ab/models/best_model.pth', map_location='cpu'))
model.eval()

# Evaluate
accuracy, metrics = evaluate_model(model, train_loader, labels)
display(f"Training set accuracy: {accuracy}")
display(f"Metrics: {metrics}")

### Inference

In [ ]:
# Path to your trained model and label mapping
model_path = 'C:/Users/don/dev/ds_test/ds_task_1ab/models/best_model.pth'
num_classes = 10

# Load the model
model = CNNModel(num_classes)
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

# Define the same transforms as used in training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load and preprocess the image
img_path = '../model_test/1.jpg'
image = Image.open(img_path).convert('RGB')
input_tensor = transform(image)

# Run inference
predicted_class_idx = predict(model, input_tensor)
display(f"Predicted class index: {predicted_class_idx}")

In [ ]:
with open('C:/Users/don/dev/ds_test/ds_task_1ab/models/stockcode_to_index.json') as f:
    stockcode_to_index = json.load(f)
index_to_stockcode = {v: k for k, v in stockcode_to_index.items()}

"Predicted StockCode:", index_to_stockcode[predicted_class_idx]